# Solution C3 · Imputation with the mean erases variance and correlation

**Type**: Break the model

## Objective

Measure, as the share of missing values grows, how imputation with the mean, deterministic regression and
regression with noise alter the variance, correlation and regression coefficient of a variable.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.experimental import enable_iterative_imputer          # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression

RHO, n = 0.8, 4000
rng = np.random.default_rng(0)
x1 = rng.normal(size=n)
x2 = RHO * x1 + np.sqrt(1 - RHO ** 2) * rng.normal(size=n)
y = x1 + x2 + rng.normal(size=n)
PS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

## Commented solution

In [ ]:
def imputations(p, seed=0):
    mask = np.random.default_rng(seed).random(n) < p
    x2_miss = np.where(mask, np.nan, x2)
    X_miss = np.column_stack([x1, x2_miss])
    return {
        "mean": SimpleImputer(strategy="mean").fit_transform(X_miss)[:, 1],
        "regression": IterativeImputer(random_state=0).fit_transform(X_miss)[:, 1],
        "with_noise": IterativeImputer(sample_posterior=True, random_state=0).fit_transform(X_miss)[:, 1],
    }


def metrics(x2_imp):
    return {"var_ratio": x2_imp.var() / x2.var(),
            "corr": np.corrcoef(x1, x2_imp)[0, 1],
            "coef_x2": LinearRegression().fit(np.column_stack([x1, x2_imp]), y).coef_[1]}

In [ ]:
rows = {}
for p in PS:
    for method, x2_imp in imputations(p).items():
        rows[(p, method)] = metrics(x2_imp)
results = pd.DataFrame(rows).T
results.index.names = ["p", "method"]
results.round(3)

In [ ]:
theoretical_var = 1 - np.array(PS)
theoretical_corr = np.sqrt(1 - np.array(PS)) * RHO
print("p                :", PS)
print("expected var     :", theoretical_var.round(2), "| measured:", results.xs("mean", level="method").var_ratio.round(2).to_numpy())
print("expected corr    :", theoretical_corr.round(2), "| measured:", results.xs("mean", level="method")["corr"].round(2).to_numpy())

## Check

In [ ]:
with verify():
    check(set(imputations(0.3)) == {"mean", "regression", "with_noise"}, "three imputation methods", "keys: mean, regression, with_noise")
    check(len(results) == 18 and set(results.columns) == {"var_ratio", "corr", "coef_x2"}, "18 rows and the three metrics", "6 values of p x 3 methods")
    mean_rows = results.xs("mean", level="method")
    check(np.all(np.abs(mean_rows.var_ratio.to_numpy() - (1 - np.array(PS))) < 0.06),
          "with the mean the variance drops by about a factor (1 - p)", "var_ratio of the mean must follow 1 - p")
    check(np.all(np.abs(mean_rows["corr"].to_numpy() - np.sqrt(1 - np.array(PS)) * RHO) < 0.06),
          "with the mean the correlation drops by about sqrt(1 - p)", "corr of the mean must follow sqrt(1 - p) * 0.8")
    check(close(theoretical_var, 1 - np.array(PS)) and close(theoretical_corr, np.sqrt(1 - np.array(PS)) * RHO),
          "correct theoretical formulas", "theoretical_var = 1 - p, theoretical_corr = sqrt(1 - p) * 0.8")
    check(mean_rows.coef_x2.loc[0.6] < mean_rows.coef_x2.loc[0.1] - 0.3, "the coefficient of x2 is attenuated by the mean, more and more as p grows",
          "coef_x2 with the mean must drop sharply from p=0.1 to p=0.6")
    reg = results.xs("regression", level="method")
    check(reg["corr"].loc[0.5] > RHO + 0.05, "deterministic regression INFLATES the correlation (the imputed values are an exact function of x1)",
          "with p=0.5 the correlation after deterministic regression should exceed 0.85")
    noisy = results.xs("with_noise", level="method")
    check(abs(noisy["corr"].loc[0.5] - RHO) < 0.06 and abs(noisy.var_ratio.loc[0.5] - 1) < 0.1,
          "with noise correlation and variance stay the true ones", "with p=0.5 with_noise should have corr about 0.8 and var_ratio about 1")
done()

## What you learned

- **Mean**: the imputed values are all equal, so they do not vary together with anything. The variance drops by a factor
  (1 − p) and the correlation with the other variables by sqrt(1 − p): with p = 0.5, from 0.8 to about 0.57. The regression coefficient
  of that variable gets **attenuated** toward zero.
- **Deterministic regression**: every imputed value is an exact function of the other columns, so the relationship looks
  stronger than the true one: the correlation goes above 0.8 and the variance stays too low.
- **Regression with noise** (`sample_posterior=True`): it adds the residual noise to the predictions and preserves both variance and
  correlation. It is the idea behind multiple imputation. Careful: the coefficient of `x2` toward `y` is still
  attenuated (in the table it drops below 0.5 with p = 0.5), because the imputation here does not know `y`: the added values are
  noise with respect to `y`. To not lose that relationship the imputation model must also include the target
  (computed only on the training set).
- For predicting (not for estimating relationships) the difference is often small; for **inference** (coefficients, correlations) the
  mean is a method to avoid.

## Common pitfall

Choosing the imputer by looking only at the error on the imputed values (RMSE, as in B2). An imputer with the lowest RMSE, like
deterministic regression, produces a distribution with the wrong variability: the right criterion depends on the use
you will make of the data.

## Interview question

<details><summary>Why is imputation with the mean criticized in statistical analyses, but often used in machine learning?</summary>

Because in statistical analysis what matters are variances, correlations and coefficients, and the mean distorts them systematically (it shortens the
variance by a factor 1 − p and weakens the correlations). In machine learning the goal is predictive performance: with
few missing values and a flexible model the distortion costs little, and the mean is simple, fast and robust. In both cases
I add the missingness indicator and check the effect with cross-validation.

</details>